# [요구사항 1] titanic 딥러닝 모델 기본 훈련

In [13]:
from pathlib import Path
from datetime import datetime
from argparse import Namespace

import pandas as pd
import torch
from torch import nn, optim
from torch.utils.data import Dataset, DataLoader, random_split
from sklearn.preprocessing import LabelEncoder
import wandb


pd.set_option("display.width", None)
pd.set_option("display.max_columns", None)

# homework_3을 기준으로 CSV가 있는 폴더를 지정한다.
DATA_PATH = Path("../homework_2")


class TitanicDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return {
            "input": self.X[idx],
            "target": self.y[idx]
        }

    def __str__(self):
        return (
            f"Data Size: {len(self.X)}, "
            f"Input Shape: {self.X.shape}, "
            f"Target Shape: {self.y.shape}"
        )


class TitanicTestDataset(Dataset):
    def __init__(self, X):
        self.X = torch.tensor(X, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return {"input": self.X[idx]}


def get_preprocessed_dataset_1(all_df):
    # 객실 등급별 평균 요금으로 Fare 결측치를 채운다.
    fare_mean = all_df.groupby("Pclass")["Fare"].transform("mean")
    all_df["Fare"] = all_df["Fare"].fillna(fare_mean)

    return all_df


def get_preprocessed_dataset_2(all_df):
    # 이름을 성, 호칭, 이름으로 분리한다.
    name_df = all_df["Name"].str.split(r"[,.]", n=2, expand=True)
    name_df.columns = ["family_name", "title", "name"]

    for column in name_df.columns:
        name_df[column] = name_df[column].str.strip()

    return pd.concat([all_df, name_df], axis=1)


def get_preprocessed_dataset_3(all_df):
    # 호칭별 나이 중앙값을 반올림하여 Age 결측치를 채운다.
    age_median = (
        all_df.groupby("title")["Age"]
        .transform("median")
        .round()
    )

    all_df["Age"] = all_df["Age"].fillna(age_median)

    return all_df


def get_preprocessed_dataset_4(all_df):
    # 본인을 제외한 동반 가족 수를 계산한다.
    all_df["family_num"] = all_df["Parch"] + all_df["SibSp"]

    # 혼자 탑승했으면 1, 동반 가족이 있으면 0으로 설정한다.
    all_df["alone"] = (all_df["family_num"] == 0).astype("float32")

    return all_df.drop(
        columns=[
            "PassengerId",
            "Name",
            "family_name",
            "name",
            "Ticket",
            "Cabin"
        ]
    )


def get_preprocessed_dataset_5(all_df):
    # 주요 호칭 네 가지 외에는 other로 묶는다.
    main_titles = ["Mr", "Miss", "Mrs", "Master"]

    all_df.loc[
        ~all_df["title"].isin(main_titles), "title"
    ] = "other"

    # inplace 대신 결과를 직접 대입하여 결측치를 채운다.
    all_df["Embarked"] = all_df["Embarked"].fillna("missing")

    return all_df


def get_preprocessed_dataset_6(all_df):
    # object와 문자열 자료형을 모두 선택하여 숫자로 변환한다.
    category_features = all_df.select_dtypes(
        include=["object", "string"]
    ).columns

    for column in category_features:
        encoder = LabelEncoder()
        all_df[column] = encoder.fit_transform(all_df[column])

    return all_df


def get_preprocessed_dataset():
    train_df = pd.read_csv(DATA_PATH / "train.csv")
    test_df = pd.read_csv(DATA_PATH / "test.csv")

    # 기존 전처리 코드와 동일하게 두 데이터를 합쳐 처리한다.
    all_df = pd.concat(
        [train_df, test_df],
        sort=False,
        ignore_index=True
    )

    all_df = get_preprocessed_dataset_1(all_df)
    all_df = get_preprocessed_dataset_2(all_df)
    all_df = get_preprocessed_dataset_3(all_df)
    all_df = get_preprocessed_dataset_4(all_df)
    all_df = get_preprocessed_dataset_5(all_df)
    all_df = get_preprocessed_dataset_6(all_df)

    print("전처리 결과:")
    print(all_df.head(10))

    # 원본 학습 데이터 개수를 기준으로 다시 분리한다.
    n_train = len(train_df)

    train_X = all_df.iloc[:n_train].drop(columns="Survived")
    train_y = train_df["Survived"]
    test_X = all_df.iloc[n_train:].drop(columns="Survived")

    # NumPy 자료형을 명시하여 텐서로 변환한다.
    dataset = TitanicDataset(
        train_X.to_numpy(dtype="float32"),
        train_y.to_numpy(dtype="int64")
    )

    print(dataset)

    # 실험마다 같은 학습·검증 데이터로 비교하도록 분할 시드를 고정한다.
    train_dataset, validation_dataset = random_split(
        dataset,
        [0.8, 0.2],
        generator=torch.Generator().manual_seed(42)
    )

    test_dataset = TitanicTestDataset(
        test_X.to_numpy(dtype="float32")
    )

    return train_dataset, validation_dataset, test_dataset


def get_data(run):
    train_dataset, validation_dataset, test_dataset = (
        get_preprocessed_dataset()
    )

    print("학습 데이터:", len(train_dataset))
    print("검증 데이터:", len(validation_dataset))
    print("테스트 데이터:", len(test_dataset))

    train_data_loader = DataLoader(
        dataset=train_dataset,
        batch_size=run.config.batch_size,
        shuffle=True
    )

    validation_data_loader = DataLoader(
        dataset=validation_dataset,
        batch_size=len(validation_dataset)
    )

    return train_data_loader, validation_data_loader


class MyModel(nn.Module):
    def __init__(self, n_input, n_output, run):
        super().__init__()

        # 교수님 코드의 은닉층 두 개와 ReLU 구조를 유지한다.
        self.model = nn.Sequential(
            nn.Linear(n_input, run.config.n_hidden_unit_list[0]),
            nn.ReLU(),
            nn.Linear(
                run.config.n_hidden_unit_list[0],
                run.config.n_hidden_unit_list[1]
            ),
            nn.ReLU(),
            nn.Linear(
                run.config.n_hidden_unit_list[1],
                n_output
            )
        )

    def forward(self, x):
        return self.model(x)


def get_model_and_optimizer(run):
    # 입력 특성 10개로 사망·생존에 대한 점수 두 개를 출력한다.
    model = MyModel(n_input=10, n_output=2, run=run)

    optimizer = optim.SGD(
        model.parameters(),
        lr=run.config.learning_rate
    )

    return model, optimizer


def training_loop(
    model,
    optimizer,
    train_data_loader,
    validation_data_loader,
    run
):
    # 생존 여부를 분류하므로 MSELoss 대신 CrossEntropyLoss를 사용한다.
    # 모델 출력에는 Softmax를 적용하지 않는다.
    loss_fn = nn.CrossEntropyLoss()

    for epoch in range(1, run.config.epochs + 1):
        model.train()

        loss_train = 0.0
        num_trains = 0

        for batch in train_data_loader:
            # TitanicDataset이 반환하는 딕셔너리에서 값을 꺼낸다.
            inputs = batch["input"]
            targets = batch["target"]

            outputs = model(inputs)
            loss = loss_fn(outputs, targets)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            # 배치별 승객 수를 반영해 평균 손실을 계산한다.
            batch_size = inputs.size(0)
            loss_train += loss.item() * batch_size
            num_trains += batch_size

        model.eval()

        loss_validation = 0.0
        num_validations = 0

        # 검증에서는 기울기 계산과 가중치 업데이트를 하지 않는다.
        with torch.no_grad():
            for batch in validation_data_loader:
                inputs = batch["input"]
                targets = batch["target"]

                outputs = model(inputs)
                loss = loss_fn(outputs, targets)

                batch_size = inputs.size(0)
                loss_validation += loss.item() * batch_size
                num_validations += batch_size

        train_loss = loss_train / num_trains
        validation_loss = loss_validation / num_validations

        # 과제에서 요구하는 두 손실을 매 Epoch 기록한다.
        run.log({
            "Epoch": epoch,
            "Training loss": train_loss,
            "Validation loss": validation_loss
        })

        if (
            epoch == 1
            or epoch % 100 == 0
            or epoch == run.config.epochs
        ):
            print(
                f"Epoch {epoch:4d}/{run.config.epochs}, "
                f"Training loss: {train_loss:.4f}, "
                f"Validation loss: {validation_loss:.4f}"
            )


def main(args):
    # 모델 초기화를 재현하기 위해 시드를 고정한다.
    torch.manual_seed(42)

    current_time_str = (
        datetime.now()
        .astimezone()
        .strftime("%Y-%m-%d_%H-%M-%S")
    )

    # 기본 훈련은 교수님 코드의 설정값을 유지한다.
    config = {
        "epochs": args.epochs,
        "batch_size": args.batch_size,
        "learning_rate": 1e-3,
        "n_hidden_unit_list": [20, 20]
    }

    with wandb.init(
        mode="online" if args.wandb else "disabled",
        project="titanic_model_training",
        notes="Titanic basic training",
        tags=["titanic", "baseline"],
        name=f"baseline_{current_time_str}",
        config=config
    ) as run:
        print(args)
        print(dict(run.config))

        # 그래프의 가로축을 Epoch로 설정한다.
        run.define_metric("Epoch")
        run.define_metric("Training loss", step_metric="Epoch")
        run.define_metric(
            "Validation loss",
            step_metric="Epoch",
            summary="min"
        )

        train_loader, validation_loader = get_data(run)
        model, optimizer = get_model_and_optimizer(run)

        print(model)

        training_loop(
            model=model,
            optimizer=optimizer,
            train_data_loader=train_loader,
            validation_data_loader=validation_loader,
            run=run
        )

    return model


# 노트북에서는 명령줄 인자 대신 설정값을 직접 지정한다.
args = Namespace(
    wandb=True,
    batch_size=512,
    epochs=1000
)

baseline_model = main(args)

# 고찰
# 은닉층의 활성화 함수만 변경하여 비선형 변환 방식에 따른 성능 차이를 비교하였습니다.
# 같은 초기 가중치, 데이터 분할, 배치 크기, 학습률을 사용하여 비교 조건을 맞췄습니다.
# 이번 배치 크기 64, 학습률 0.001 조건에서는 ELU의 최소 검증 손실이 가장 낮았습니다.
#
# ELU는 음수 구간에도 기울기가 있지만, 이 특성만으로 이번 결과의 원인을
# 단정할 수는 없습니다. 활성화 함수의 일반적인 장점과 실제 실험 결과를
# 구분하고, 주어진 데이터에서 직접 성능을 확인해야 한다는 것을 배웠습니다.

Namespace(wandb=True, batch_size=512, epochs=1000)
{'epochs': 1000, 'batch_size': 512, 'learning_rate': 0.001, 'n_hidden_unit_list': [20, 20]}
전처리 결과:
   Survived  Pclass  Sex   Age  SibSp  Parch     Fare  Embarked  title  \
0       0.0       3    1  22.0      1      0   7.2500         2      2   
1       1.0       1    0  38.0      1      0  71.2833         0      3   
2       1.0       3    0  26.0      0      0   7.9250         2      1   
3       1.0       1    0  35.0      1      0  53.1000         2      3   
4       0.0       3    1  35.0      0      0   8.0500         2      2   
5       0.0       3    1  29.0      0      0   8.4583         1      2   
6       0.0       1    1  54.0      0      0  51.8625         2      2   
7       0.0       3    1   2.0      3      1  21.0750         2      0   
8       1.0       3    0  27.0      0      2  11.1333         2      3   
9       1.0       2    0  14.0      1      0  30.0708         0      3   

   family_num  alone  
0          

Epoch,▁▁▁▂▂▂▂▂▂▂▃▃▃▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇▇▇███
Training loss,█▇▆▄▄▄▄▄▃▃▃▃▃▃▃▂▂▂▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁
Validation loss,█▅▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▁▁▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
Epoch,1000
Training loss,0.58752


![Training loss와 Validation loss](baseline_los0s.png)

# [요구사항 2] Hyper Parameter 변경 및 선택하기

In [8]:
# 활성화 함수는 신경망이 단순한 직선 관계를 넘어 복잡한 관계를 학습하게 한다.
# 활성화 함수만 비교하기 위해 배치 크기 64, 학습률 0.001,
# 은닉층 [20, 20], SGD, 1000 Epoch 조건을 동일하게 유지한다.

# Sigmoid
# 입력값을 0~1 범위로 변환한다.
# 장점: 출력 범위가 일정하며 함수가 부드럽게 연결된다.
# 단점: 입력의 절댓값이 크면 기울기가 매우 작아져 학습이 느려질 수 있다.
# 현재 데이터는 나이와 요금 등의 입력 크기가 달라 이러한 영향을 받을 수 있다.
# 여기서는 생존 확률을 출력하는 용도가 아니라 은닉층의 활성화 함수로 사용한다.

# ReLU
# 양수 입력은 그대로 출력하고, 음수 입력은 0으로 만든다.
# 장점: 계산이 간단하고 양수 구간에서는 기울기가 작아지는 문제를 줄인다.
# 단점: 뉴런이 계속 음수 입력만 받으면 기울기가 0이 되어 학습이 멈출 수 있다.

# ELU
# 양수 입력은 그대로 출력하고, 음수 입력에는 지수함수를 적용한다.
# 장점: 음수 구간에도 기울기가 있어 ReLU의 뉴런 학습 중단 문제를 줄일 수 있다.
# 단점: 지수 계산이 필요하고, 매우 큰 음수에서는 기울기가 작아진다.

# LeakyReLU
# 양수 입력은 그대로 출력하고, 음수 입력에는 작은 기울기를 적용한다.
# 이 코드에서 사용하는 기본 음수 기울기는 0.01이다.
# 장점: 음수 구간에서도 기울기가 유지되어 ReLU의 단점을 완화한다.
# 단점: 음수 기울기 설정에 따라 결과가 달라지며 ReLU보다 항상 좋은 것은 아니다.

# 가장 좋은 함수는 일반적인 장점만으로 정하지 않는다.
# 이번 실험에서 최소 Validation loss가 가장 낮은 함수를 선택한다.


# 요구사항 1에서 정의한 전처리 함수로 데이터를 한 번만 준비한다.
# 모든 활성화 함수에 동일한 학습·검증 데이터를 사용한다.
train_dataset, validation_dataset, test_dataset = (
    get_preprocessed_dataset()
)


class ComparisonModel(nn.Module):
    def __init__(self, activation):
        super().__init__()

        activation_class = {
            "Sigmoid": nn.Sigmoid,
            "ReLU": nn.ReLU,
            "ELU": nn.ELU,
            "LeakyReLU": nn.LeakyReLU
        }[activation]

        # 두 은닉층에 선택한 활성화 함수를 적용한다.
        self.model = nn.Sequential(
            nn.Linear(10, 20),
            activation_class(),
            nn.Linear(20, 20),
            activation_class(),
            nn.Linear(20, 2)
        )

    def forward(self, x):
        return self.model(x)


# 다음 셀의 배치 크기·학습률 비교에서도 사용할 공통 훈련 함수이다.
def compare_training(activation, batch_size, learning_rate, group):
    # 실험마다 동일한 초기 가중치를 사용한다.
    torch.manual_seed(42)

    train_loader = DataLoader(
        dataset=train_dataset,
        batch_size=batch_size,
        shuffle=True,
        generator=torch.Generator().manual_seed(42)
    )

    validation_loader = DataLoader(
        dataset=validation_dataset,
        batch_size=len(validation_dataset)
    )

    model = ComparisonModel(activation)
    optimizer = optim.SGD(model.parameters(), lr=learning_rate)
    loss_fn = nn.CrossEntropyLoss()

    best_loss = float("inf")

    with wandb.init(
        project="titanic_model_training",
        group=group,
        name=f"{activation}_batch{batch_size}_lr{learning_rate:g}",
        config={
            "activation": activation,
            "batch_size": batch_size,
            "learning_rate": learning_rate,
            "epochs": 1000
        }
    ) as run:
        # 손실 그래프의 가로축을 Epoch로 설정한다.
        run.define_metric("Epoch")
        run.define_metric("Training loss", step_metric="Epoch")
        run.define_metric(
            "Validation loss",
            step_metric="Epoch",
            summary="min"
        )

        for epoch in range(1, 1001):
            model.train()
            train_loss_sum = 0.0

            for batch in train_loader:
                inputs = batch["input"]
                targets = batch["target"]

                outputs = model(inputs)
                loss = loss_fn(outputs, targets)

                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

                # 배치 크기를 반영하여 승객 수 기준 평균 손실을 구한다.
                train_loss_sum += loss.item() * inputs.size(0)

            model.eval()
            validation_loss_sum = 0.0

            # 검증에서는 가중치를 수정하지 않는다.
            with torch.no_grad():
                for batch in validation_loader:
                    inputs = batch["input"]
                    targets = batch["target"]

                    outputs = model(inputs)
                    loss = loss_fn(outputs, targets)

                    validation_loss_sum += (
                        loss.item() * inputs.size(0)
                    )

            train_loss = train_loss_sum / len(train_dataset)
            validation_loss = (
                validation_loss_sum / len(validation_dataset)
            )

            run.log({
                "Epoch": epoch,
                "Training loss": train_loss,
                "Validation loss": validation_loss
            })

            # 마지막 Epoch의 손실 대신 훈련 중 최소 검증 손실을 비교한다.
            best_loss = min(best_loss, validation_loss)

    return {
        "activation": activation,
        "batch_size": batch_size,
        "learning_rate": learning_rate,
        "best_validation_loss": best_loss
    }


# 활성화 함수만 변경하고 나머지 조건은 고정한다.
activation_results = []

for activation in ["Sigmoid", "ReLU", "ELU", "LeakyReLU"]:
    activation_results.append(
        compare_training(
            activation=activation,
            batch_size=64,
            learning_rate=1e-3,
            group="activation_comparison"
        )
    )

# 검증 손실이 낮은 순서대로 비교 결과를 출력한다.
activation_df = (
    pd.DataFrame(activation_results)
    .sort_values("best_validation_loss")
    .reset_index(drop=True)
)

display(activation_df)

# 선택한 함수는 다음 배치 크기 비교에서 사용한다.
best_activation = activation_df.iloc[0]["activation"]
best_activation_loss = activation_df.iloc[0]["best_validation_loss"]

print(
    f"배치 크기 64, 학습률 0.001 조건에서는 "
    f"{best_activation}이 가장 좋은 결과를 보였다."
)
print(
    f"선택 이유: 최소 Validation loss가 "
    f"{best_activation_loss:.4f}로 네 함수 중 가장 낮았다."
)

전처리 결과:
   Survived  Pclass  Sex   Age  SibSp  Parch     Fare  Embarked  title  \
0       0.0       3    1  22.0      1      0   7.2500         2      2   
1       1.0       1    0  38.0      1      0  71.2833         0      3   
2       1.0       3    0  26.0      0      0   7.9250         2      1   
3       1.0       1    0  35.0      1      0  53.1000         2      3   
4       0.0       3    1  35.0      0      0   8.0500         2      2   
5       0.0       3    1  29.0      0      0   8.4583         1      2   
6       0.0       1    1  54.0      0      0  51.8625         2      2   
7       0.0       3    1   2.0      3      1  21.0750         2      0   
8       1.0       3    0  27.0      0      2  11.1333         2      3   
9       1.0       2    0  14.0      1      0  30.0708         0      3   

   family_num  alone  
0           1    0.0  
1           1    0.0  
2           0    1.0  
3           1    0.0  
4           0    1.0  
5           0    1.0  
6           0   

Epoch,▁▁▁▁▁▂▂▂▂▂▃▃▃▃▃▃▃▄▄▄▄▄▄▅▅▆▆▆▆▆▆▆▇▇▇▇████
Training loss,█▇▇▆▆▆▆▆▆▆▅▅▅▅▅▅▅▅▄▄▄▃▃▃▃▃▃▃▃▃▂▂▂▂▂▁▁▁▁▁
Validation loss,██▆▅▅▄▄▄▄▄▄▃▃▃▃▃▃▃▃▃▃▂▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁
Epoch,1000
Training loss,0.63053


Epoch,▁▁▁▂▂▃▃▃▃▃▄▄▄▄▄▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▆▆▆▇█████
Training loss,██▇▇▇▇▆▆▆▆▆▅▅▅▅▅▅▄▃▃▃▃▃▃▃▃▃▃▃▂▂▂▂▂▂▁▁▁▁▁
Validation loss,███▇█▇▇▇▇▆▇▆▆▆▆▅▅▆▅▄▄▅▄▄▅▄▅▄▄▄▃▄▂▄▃▄▆▃▂▁
Epoch,1000
Training loss,0.54498


Epoch,▁▁▁▁▁▁▂▂▂▂▂▃▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇▇▇▇▇████
Training loss,██▇█▇▇▇▇▇▇▆▆▆▆▆▆▆▆▆▅▅▅▅▅▅▄▄▄▄▄▄▃▃▃▃▃▃▂▂▁
Validation loss,██▇▇▇▆▇▇▆▅▆▅▅▅▅▅▄▆▄▅▅▄▄▄▅▄▂▂▄▅▂▃▃▄▄▃▄▃▁█
Epoch,1000
Training loss,0.51956


Epoch,▁▁▁▂▂▂▂▂▂▂▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▅▅▆▆▆▆▇▇▇██████
Training loss,█▇▇▆▆▆▆▆▅▅▅▅▄▄▄▄▄▄▄▄▄▄▄▄▃▃▃▃▂▂▂▂▂▂▂▁▁▁▁▁
Validation loss,█▇▇█▇▇▇▆▆▇▇▆▆▆▆▆▅▅▆▅▅▅▅▄▅▄▄▃▃▅▄▃▄▂▃▂▁▃▂▃
Epoch,1000
Training loss,0.54654


,activation,batch_size,learning_rate,best_validation_loss
0,ELU,64,0.001,0.489888
1,ReLU,64,0.001,0.522270
2,LeakyReLU,64,0.001,0.522874
3,Sigmoid,64,0.001,0.613784


배치 크기 64, 학습률 0.001 조건에서는 ELU이 가장 좋은 결과를 보였다.
선택 이유: 최소 Validation loss가 0.4899로 네 함수 중 가장 낮았다.


In [14]:
# 배치 크기는 가중치를 한 번 업데이트할 때 사용하는 데이터의 개수이다.
# 앞에서 선택한 활성화 함수와 학습률 0.001을 고정하고 배치 크기만 비교한다.
# 모든 실험은 동일한 학습·검증 데이터와 1000 Epoch를 사용한다.

# 배치 크기 32
# 학습 데이터를 32명씩 묶어 가중치를 업데이트한다.
# 장점: 가중치를 자주 업데이트하며, 배치별 기울기의 변동이
# 새로운 데이터에 대한 성능 향상에 도움이 되는 경우가 있다.
# 단점: 업데이트 횟수가 많아 실행 시간이 늘고 손실이 흔들릴 수 있다.

# 배치 크기 64
# 학습 데이터를 64명씩 묶어 가중치를 업데이트한다.
# 장점: 배치 32보다 업데이트 횟수가 적고, 128보다 자주 업데이트한다.
# 단점: 중간 크기라는 이유만으로 가장 좋은 성능을 보장하지는 않는다.

# 배치 크기 128
# 학습 데이터를 128명씩 묶어 가중치를 업데이트한다.
# 장점: 한 번에 더 많은 데이터로 기울기를 계산하며,
# 한 Epoch의 업데이트 횟수가 적어 실행 시간이 줄어들 수 있다.
# 단점: 배치당 메모리 사용량이 늘고, 동일한 Epoch에서 학습이 부족할 수 있다.

# 학습 데이터가 713개일 때 한 Epoch의 업데이트 횟수는
# 배치 32에서 23회, 64에서 12회, 128에서 6회이다.
# 따라서 이번 결과는 같은 업데이트 횟수가 아닌 같은 Epoch 수 기준의 비교이다.


batch_results = []

for batch_size in [32, 64, 128]:
    batch_results.append(
        compare_training(
            activation=best_activation,
            batch_size=batch_size,
            learning_rate=1e-3,
            group="batch_size_comparison"
        )
    )

# 최소 검증 손실이 낮은 순서대로 결과를 출력한다.
batch_df = (
    pd.DataFrame(batch_results)
    .sort_values("best_validation_loss")
    .reset_index(drop=True)
)

display(batch_df)

# 선택한 배치 크기는 다음 학습률 비교에서 사용한다.
best_batch_size = int(batch_df.iloc[0]["batch_size"])
best_batch_loss = batch_df.iloc[0]["best_validation_loss"]

print(
    f"활성화 함수 {best_activation}, 학습률 0.001 조건에서는 "
    f"배치 크기 {best_batch_size}가 가장 좋은 결과를 보였다."
)
print(
    f"선택 이유: 최소 Validation loss가 "
    f"{best_batch_loss:.4f}로 세 배치 크기 중 가장 낮았다."
)

# 고찰
# ELU와 학습률 0.001을 고정하고 배치 크기 32, 64, 128을 비교하였습니다.
# 이번 실험에서는 배치 크기 32의 최소 검증 손실이 가장 낮았습니다.
#
# 배치가 작으면 한 Epoch에 가중치를 더 자주 업데이트하므로,
# 동일한 1000 Epoch라도 배치 크기에 따라 총 업데이트 횟수가 달라집니다.
# 따라서 이번 결과는 배치 크기뿐 아니라 업데이트 횟수 차이의 영향도 포함합니다.
# 작은 배치가 항상 우수하다고 일반화하기보다 실험 조건과 함께 해석해야 합니다.

Epoch,▁▁▁▁▂▂▂▂▂▂▂▂▂▃▃▃▃▃▃▃▃▄▄▄▄▄▅▅▅▆▆▆▇▇▇▇▇▇▇█
Training loss,███▇▇▇▇▇▇▇▇▇▆▆▆▆▆▆▆▆▆▅▅▅▅▄▄▄▃▂▂▂▂▃▁▁▂▂▁▁
Validation loss,█████▇▇▇▆▇▇▇▆▆▆▆▆▆█▆▄▄▅▅▄▅▅▅▄▄▃▂▂▁▁▁▁▃▁▂
Epoch,1000
Training loss,0.45013


Epoch,▁▁▁▁▁▁▂▂▂▂▃▃▃▃▄▄▄▅▅▅▅▅▅▅▅▆▆▆▆▆▆▆▆▆▆▇▇███
Training loss,███▇▇▇▇▆▆▆▆▆▆▆▆▆▅▅▆▅▅▅▅▅▅▅▅▅▄▄▄▄▄▄▄▃▃▃▃▁
Validation loss,████▇▇▇▇▇▇▇▆▆▆▆▇▅▆▅▆▄▆▅▅▄▄▇▄▄▆▅▃▅▅▂▃▁▂▁▁
Epoch,1000
Training loss,0.51956


Epoch,▁▁▁▁▂▂▂▂▂▃▃▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▆▆▆▆▆▆▇▇▇▇████
Training loss,█▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
Validation loss,█▇▇▇▇▆▆▆▆▅▅▄▄▄▄▄▄▃▄▃▃▃▃▃▃▃▂▃▃▂▂▂▂▂▂▁▁▂▁▁
Epoch,1000
Training loss,0.5609


,activation,batch_size,learning_rate,best_validation_loss
0,ELU,32,0.001,0.394084
1,ELU,64,0.001,0.489888
2,ELU,128,0.001,0.550292


활성화 함수 ELU, 학습률 0.001 조건에서는 배치 크기 32가 가장 좋은 결과를 보였다.
선택 이유: 최소 Validation loss가 0.3941로 세 배치 크기 중 가장 낮았다.


In [ ]:
# 학습률은 계산된 기울기를 가중치 변경에 얼마나 반영할지 결정한다.
# 앞에서 선택한 활성화 함수와 배치 크기를 고정하고 학습률만 비교한다.
# 모든 실험은 동일한 학습·검증 데이터와 1000 Epoch를 사용한다.

# 학습률 0.01 (1e-2)
# 가중치를 비교적 큰 폭으로 변경한다.
# 장점: 손실을 빠르게 줄여 적은 Epoch에 수렴할 수 있다.
# 단점: 좋은 가중치를 지나치거나 손실이 흔들리고 발산할 수 있다.
# 현재처럼 입력을 표준화하지 않은 경우 큰 입력값의 영향도 받을 수 있다.

# 학습률 0.001 (1e-3)
# 비교하는 세 값 중 중간 크기로 가중치를 변경한다.
# 장점: 0.01보다 작은 폭으로 조정하여 큰 진동을 줄일 수 있다.
# 단점: 0.01보다 학습이 느릴 수 있으며 항상 가장 좋은 값은 아니다.

# 학습률 0.0001 (1e-4)
# 가중치를 작은 폭으로 변경한다.
# 장점: 큰 폭의 가중치 변경으로 좋은 지점을 지나칠 가능성을 줄인다.
# 단점: 학습이 느려 1000 Epoch 안에 충분히 학습하지 못할 수 있다.

# 학습률은 작을수록 또는 클수록 무조건 좋은 것이 아니다.
# 이번 조건에서 최소 Validation loss가 가장 낮은 값을 선택한다.


learning_rate_results = []

for learning_rate in [1e-2, 1e-3, 1e-4]:
    learning_rate_results.append(
        compare_training(
            activation=best_activation,
            batch_size=best_batch_size,
            learning_rate=learning_rate,
            group="learning_rate_comparison"
        )
    )

# 최소 검증 손실이 낮은 순서대로 결과를 출력한다.
learning_rate_df = (
    pd.DataFrame(learning_rate_results)
    .sort_values("best_validation_loss")
    .reset_index(drop=True)
)

display(learning_rate_df)

best_learning_rate = float(
    learning_rate_df.iloc[0]["learning_rate"]
)
best_learning_rate_loss = (
    learning_rate_df.iloc[0]["best_validation_loss"]
)

print(
    f"활성화 함수 {best_activation}, 배치 크기 {best_batch_size} 조건에서는 "
    f"학습률 {best_learning_rate:g}가 가장 좋은 결과를 보였다."
)
print(
    f"선택 이유: 최소 Validation loss가 "
    f"{best_learning_rate_loss:.4f}로 세 학습률 중 가장 낮았다."
)

# 요구사항 3에서 사용할 최종 하이퍼파라미터를 저장한다.
best_config = {
    "activation": best_activation,
    "batch_size": best_batch_size,
    "learning_rate": best_learning_rate
}

print("\n최종 선택한 하이퍼파라미터")
print("활성화 함수:", best_config["activation"])
print("배치 크기:", best_config["batch_size"])
print("학습률:", best_config["learning_rate"])

# 고찰
# ELU와 배치 크기 32를 고정하고 학습률 0.01, 0.001, 0.0001을 비교하였습니다.
# 최소 검증 손실은 각각 약 0.3646, 0.3941, 0.5788로,
# 이번 조건에서는 학습률 0.01이 가장 좋은 결과를 보였습니다.
#
# 학습률은 작을수록 좋은 것이 아니라 정해진 훈련 횟수 안에서
# 충분히 학습하면서도 지나치게 불안정해지지 않는 값을 선택해야 합니다.
# 이번에는 ELU, 배치 크기 32, 학습률 0.01을 최종 설정으로 선택하였습니다.
# 다만 단계별 비교이므로 전체 조합 중 최적 설정이라고 단정할 수는 없습니다.

Epoch,▁▁▁▂▂▃▃▃▃▃▄▄▄▅▅▅▅▅▅▅▅▅▅▅▅▆▆▆▇▇▇▇▇▇▇▇████
Training loss,█▇█▇▇▇▆▅▅▅▄▄▂▃▃▂▅▂▃▂▂▂▂▃▂▃▂▂▂▂▂▂▂▂▂▁▂▁▁▁
Validation loss,▂▃▄▂▂▂▂▂▂▂▁▃▂▁▁▁▃▂▂▁▂▂▂▁▁▁█▁▂▁▁▁▁▂▂▁▁▃▁▁
Epoch,1000
Training loss,0.40599


Epoch,▁▁▁▁▁▁▂▂▂▂▂▃▃▃▃▃▃▃▄▄▄▄▅▅▅▆▆▆▆▆▆▇▇▇▇▇████
Training loss,█▃▃▃▃▃▃▃▃▃▃▃▃▃▃▃▃▃▂▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁
Validation loss,▆▇▆▆▆▇▇▆▆▆▆▅▅▆▆▅▄▅▄▇▅▃▃▃▃█▂█▂▂▂▁▂▃▁▁▅▁▁▂
Epoch,1000
Training loss,0.45013


# [요구사항 3] 테스트 및 submission.csv 생성

In [ ]:
from copy import deepcopy


# 요구사항 2에서 준비한 데이터와 선택한 하이퍼파라미터를 사용한다.
torch.manual_seed(42)

train_data_loader = DataLoader(
    dataset=train_dataset,
    batch_size=best_config["batch_size"],
    shuffle=True,
    generator=torch.Generator().manual_seed(42)
)

validation_data_loader = DataLoader(
    dataset=validation_dataset,
    batch_size=len(validation_dataset)
)

# 테스트 데이터는 승객 번호와 예측값의 순서를 맞추기 위해 섞지 않는다.
test_data_loader = DataLoader(
    dataset=test_dataset,
    batch_size=best_config["batch_size"],
    shuffle=False
)

model = ComparisonModel(best_config["activation"])

optimizer = optim.SGD(
    model.parameters(),
    lr=best_config["learning_rate"]
)

loss_fn = nn.CrossEntropyLoss()

epochs = 1000
patience = 100
wait = 0

best_loss = float("inf")
best_epoch = 0
best_model_state = deepcopy(model.state_dict())


# Early Stopping은 검증 손실이 일정 기간 개선되지 않으면 훈련을 멈춘다.
# 장점: 불필요한 훈련을 줄이고 과적합을 완화할 수 있다.
# 단점: 대기 기간이 너무 짧으면 이후 개선될 가능성이 있는 모델도 일찍 멈춘다.
# 여기서는 100 Epoch 동안 검증 손실이 개선되지 않으면 중단한다.

with wandb.init(
    project="titanic_model_training",
    name="final_model_early_stopping",
    config={
        **best_config,
        "epochs": epochs,
        "patience": patience
    }
) as run:
    run.define_metric("Epoch")
    run.define_metric("Training loss", step_metric="Epoch")
    run.define_metric(
        "Validation loss",
        step_metric="Epoch",
        summary="min"
    )

    for epoch in range(1, epochs + 1):
        model.train()
        train_loss_sum = 0.0

        for batch in train_data_loader:
            inputs = batch["input"]
            targets = batch["target"]

            outputs = model(inputs)
            loss = loss_fn(outputs, targets)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            train_loss_sum += loss.item() * inputs.size(0)

        model.eval()
        validation_loss_sum = 0.0

        with torch.no_grad():
            for batch in validation_data_loader:
                inputs = batch["input"]
                targets = batch["target"]

                outputs = model(inputs)
                loss = loss_fn(outputs, targets)

                validation_loss_sum += loss.item() * inputs.size(0)

        train_loss = train_loss_sum / len(train_dataset)
        validation_loss = (
            validation_loss_sum / len(validation_dataset)
        )

        run.log({
            "Epoch": epoch,
            "Training loss": train_loss,
            "Validation loss": validation_loss
        })

        if validation_loss < best_loss:
            best_loss = validation_loss
            best_epoch = epoch

            # 이후 훈련으로 값이 바뀌지 않도록 최적 가중치를 복사한다.
            best_model_state = deepcopy(model.state_dict())
            wait = 0
        else:
            wait += 1

        if wait >= patience:
            print(f"Early Stopping: Epoch {epoch}에서 훈련 중단")
            break


# 훈련이 끝난 시점이 아니라 검증 손실이 가장 낮았던 가중치를 복원한다.
model.load_state_dict(best_model_state)

print("최종 하이퍼파라미터:", best_config)
print("훈련 종료 Epoch:", epoch)
print("선택한 모델의 Epoch:", best_epoch)
print(f"최소 Validation loss: {best_loss:.4f}")


# 정답이 없는 테스트 데이터는 모델 선택에 사용하지 않고 최종 예측에 사용한다.
model.eval()
predictions = []

with torch.no_grad():
    for batch in test_data_loader:
        outputs = model(batch["input"])

        # 사망(0), 생존(1) 중 점수가 높은 클래스를 선택한다.
        predictions.extend(outputs.argmax(dim=1).tolist())


# 전처리에서 제거한 승객 번호를 원본 test.csv에서 가져온다.
test_df = pd.read_csv(DATA_PATH / "test.csv")

submission = pd.DataFrame({
    "PassengerId": test_df["PassengerId"].to_numpy(),
    "Survived": predictions
})

# 현재 작업 폴더에 Kaggle 제출 파일을 생성한다.
# index=False로 불필요한 인덱스 열이 저장되지 않도록 한다.
submission_path = Path.cwd() / "submission.csv"
submission.to_csv(submission_path, index=False)

display(submission.head(10))
print("제출 데이터 개수:", len(submission))
print("저장 위치:", submission_path)

# 고찰
# 마지막 Epoch의 모델이 가장 좋은 성능을 보장하지 않으므로,
# 검증 손실이 가장 낮았던 시점의 가중치를 복사하여 보관하였습니다.
# 검증 손실이 100 Epoch 동안 개선되지 않아 575 Epoch에서 훈련이 중단되었고,
# 475 Epoch의 가중치를 복원하였습니다. 이때 검증 손실은 약 0.3755였습니다.
#
# Early Stopping은 불필요한 훈련을 줄일 수 있지만,
# 대기 기간 이후에 발생할 추가 개선을 놓칠 가능성도 있습니다.
# 따라서 대기 기간 역시 학습 곡선과 함께 검토할 필요가 있습니다.
#
# 테스트 데이터는 정답이 없으므로 모델 선택에 사용하지 않았습니다.
# 예측값과 승객 번호의 순서를 맞추기 위해 테스트 데이터를 섞지 않았으며,
# PassengerId와 Survived 두 열로 구성된 418행의 submission.csv를 생성하였습니다.

Early Stopping: Epoch 575에서 훈련 중단


Epoch,▁▁▁▁▁▂▂▂▂▂▃▃▃▃▃▄▄▄▄▅▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇████
Training loss,█▇▆▆▆▄▄▄▄▄▃▅▅▃▄▃▃▃▃▃▃▃▂▂▃▂▂▂▃▁▁▂▂▂▁▁▂▁▂▃
Validation loss,▅▄█▄▄▄▄▄▄▃▃▃▃▄▃▅▄▂▄▂▆▂▂▁▂▂▂▂▂▁▇▃▂▂▄▁▁▂▁▃
Epoch,575
Training loss,0.4262


최종 하이퍼파라미터: {'activation': 'ELU', 'batch_size': 32, 'learning_rate': 0.01}
훈련 종료 Epoch: 575
선택한 모델의 Epoch: 475
최소 Validation loss: 0.3755


,PassengerId,Survived
0,892,0
1,893,0
2,894,0
3,895,0
4,896,1
5,897,0
6,898,1
7,899,0
8,900,1
9,901,0


제출 데이터 개수: 418
저장 위치: c:\Users\lji08\Desktop\딥러닝\_03_homeworks\homework_2\submission.csv


### 고찰

마지막 Epoch의 모델이 가장 좋은 성능을 보장하지는 않으므로,
검증 손실이 가장 낮았던 Epoch의 가중치를 저장하였습니다.
검증 손실이 100 Epoch 동안 개선되지 않으면 훈련을 중단하고,
저장한 최적 가중치를 복원하여 테스트 데이터의 생존 여부를 예측하였습니다.
최대 1000 Epoch에 먼저 도달하면 해당 시점에 훈련을 종료하도록 하였습니다.

테스트 데이터에는 생존 정답이 없으므로 모델 선택에는 사용하지 않았으며,
최종 예측값을 원본 승객 번호와 연결하여 submission.csv를 생성하였습니다.

In [ ]:
submission_df = pd.read_csv(submission_path)

with pd.option_context("display.max_rows", None):
    display(submission_df)

,PassengerId,Survived
0,892,0
1,893,0
2,894,0
3,895,0
4,896,1
5,897,0
6,898,1
7,899,0
8,900,1
9,901,0


# [요구사항 4] submission.csv 제출 및 등수확인

- Kaggle 점수: 0.76555
- Leaderboard 순위: 실제 순위 입력

![Kaggle 점수](https://github.com/Jaeikk/titanic-homework/blob/main/score.png?raw=true)

![Leaderboard 순위](https://github.com/Jaeikk/titanic-homework/blob/main/ranking.png?raw=true)

### 고찰

생성한 submission.csv를 Kaggle에 제출하여 약 76.56%의 평가 정확도를 확인하였습니다. 검증 손실과 Kaggle 정확도는 평가 데이터와 지표가 다르므로 숫자를 직접 비교할 수 없다는 점을 이해하였습니다.

추후에는 입력 특성의 표준화와 여러 시드에서의 반복 실험을 통해 성능과 결과의 일관성을 확인하고 싶습니다. 또한 더 엄밀한 평가를 위해 학습 데이터에서만 전처리 기준을 구하고 검증·테스트 데이터에 적용할 필요가 있습니다.

## 숙제 후기

이번 과제를 통해 데이터 전처리부터 모델 훈련, 하이퍼파라미터 비교, Early Stopping, Kaggle 제출까지의 과정을 경험하였습니다. 처음에는 코드 실행에 집중하였지만, 실험을 진행하면서 설정을 변경하는 이유와 성능을 비교하는 기준을 이해하게 되었습니다.

전처리 오류를 해결하면서 입력 데이터의 자료형과 결측치 처리도 모델 학습에 중요하다는 것을 느꼈습니다. 직접 생성한 예측 파일이 Kaggle에서 채점되는 과정을 확인하여 학습 결과를 구체적으로 체감할 수 있었습니다. 앞으로는 전처리와 모델 구성을 개선하여 예측 성능을 높여 보고 싶습니다.